# Explainable AI
---

Explainable AI (XAI) refers to techniques, methods, and frameworks that make the predictions, decisions, and behaviors of AI systems understandable to humans. As AI becomes more complex (e.g., deep learning, ensemble models), understanding "why" a model made a certain decision is crucial for trust, accountability, safety, and fairness

Purpose: User Trust, Debugging, Ethics, Regulatorics, Safety

__Categories__
1. Model-Specific vs Model-Agnostic
    - Model-specific<br>require access to internal structure (e.g., decision trees, attention maps)
    - Model-agnostic<br>work with any black-box model via inputs/outputs (e.g., LIME, SHAP, ALE)<br><br>
2. Global vs Local Explanations
    - Global: Explain the overall model behavior (e.g., feature importance, PDP, ALE)
    - Local: Explain a single prediction (e.g., LIME, SHAP, counterfactuals, ICE)<br><br>
3. Post-Hoc vs Intrinsic
    - Post-hoc: Explanations are generated after model training
    - Intrinsic: Models that are interpretable by design (e.g., linear models, decision trees)<br><br>



|Technique| Description|Type|
| :--------------------------------- |:--------------------------------------------------------------------------------- |:-------------- |
| **Feature Importance**            | Ranks how much each feature contributes to predictions                            | Global         |
| **Partial Dependence Plot (PDP)** | Shows average effect of a feature                                                 | Global         |
| **ICE & d-ICE**                   | Individual Conditional Expectation — shows how prediction changes per observation | Local          |
| **ALE**                           | Accumulated Local Effects — unbiased average effect in data-supported regions     | Global         |
| **LIME**                          | Local interpretable model-agnostic explanations via surrogate models              | Local          |
| **SHAP**                          | Shapley values for consistent, theoretically grounded feature attribution         | Local + Global |
| **Anchors**                       | IF-THEN rules that "anchor" predictions with high precision                       | Local          |
| **Counterfactuals**               | "What would need to change for a different prediction?"                           | Local          |
| **Saliency maps, Attention**      | Visualization for interpreting deep learning models                               | Model-specific |





## Prediction Anchors (2018)
---
[[paper]](https://homes.cs.washington.edu/~marcotcr/aaai18.pdf)<br>

Idea: approximate decision boundaries by simple decision rules

Anchor or Rule = a single partition of the feature space (for example, node of a decision tree) which is intened to be simple - with as few features as possible

For a given observation:
1. Find a set of candidate rules where target is stable<br>aka class distribution should be as non-uniform as possible
2. Generate random changes to the observation by perturbing features not included in the rule 
3. Evaluate how stable is the answer<br>if target won't change for most cases - a rule is a good explanation

Metrics<br>
Precision = rate of observations where answer did not change<br>
Coverage = rate of observations where rule applies

# Counterfactual explanations
---
Counterfactual observations = synthetic data points (usually close to the original point) that exemplifies the shortest path   to get the desired target<br>

Example: your credit application was rejected. GDPR gives you right to  appeal or at least understand: "What should I do to be approved?"

<img src="img/counterfactual.png" width=500>

# Watcher (2017)
---
[[paper]](https://arxiv.org/abs/1711.00399)<br>A method to find the best counterfactual explanation: for a given observation $x$ find a minimal change $x' = x+\delta$ that achieves the desired target. We find such data point by solving a minimization problem:

$$
\min_{x'} \ \lambda \cdot \text{loss}(f(x'), y_{\text{target}}) + d(x, x')
$$

Here loss function = distance to the desired class $y_{target}$, for binary problem could be $\{0,1\}$ <br>d = distance from the data point $x$ <br>$\lambda$ is a balance coefficient between proximity and validity

# DiCE (2019)
---
[[paper]](https://arxiv.org/abs/1905.07697)<br>
DiCE = Diverse Counterfactual Explanations. 

Input = trained model f(x). For a given observation find a set of minimum alterations to this data point that would change the prediction to the desired one (for example, from y=0 to y=1)

Unlike counterfactual explanations (where you return one shortest path) here we output a "ray" of diverse possible changes

Under DiCE we find such paths through solving an optimization problem. We require that paths
- must lead to the desired output
- must be short
- must be diverse
- respect the constraints (be inside original data distribution)

Formula for the optimization problem:
$$
\min_{\{x_i'\}} \sum_{i=1}^{k} \left[ \lambda_1 \cdot \text{loss}(f(x_i'), y_{\text{target}}) + \lambda_2 \cdot d(x_i', x) \right] - \lambda_3 \cdot \text{diversity}(\{x_i'\})
$$

# Contrastive Explanations Method (2018)
[[paper]](https://arxiv.org/abs/1802.07623)<br>
CEM is a method that seeks for two feature sets that for a given observation explain its target value most. First set = "Pertinent Positives", features that must be present to justify the model’s prediction. Second set = "Pertinent Negatives", 
features that must be absent to justify the same prediction

For the PN
$$
\min_{\boldsymbol{\delta}} \ c \cdot \max \left\{ 0, \max_{i \neq t} f(\mathbf{x} + \boldsymbol{\delta})_i - f(\mathbf{x} + \boldsymbol{\delta})_t + \kappa \right\} + \lambda_1 \cdot \|\boldsymbol{\delta}\|_1 + \lambda_2 \cdot \|\boldsymbol{\delta}\|_2^2 + \lambda_3 \cdot \|\mathbf{x} + \boldsymbol{\delta} - AE(\mathbf{x} + \boldsymbol{\delta})\|_2^2
$$

Our perturbation $\delta$ must change the current class $t$. So we model the distance to the closest other class as ($k$ plays a role of threshold):
$$\max \left\{ 0, f(\mathbf{x} + \boldsymbol{\delta})_t - \max_{i \neq t} f(\mathbf{x} + \boldsymbol{\delta})_i + \kappa \right\}$$

Next we add two elasticnet regularizers
$$\lambda_1 \cdot \|\boldsymbol{\delta}\|_1 + \lambda_2 \cdot \|\boldsymbol{\delta}\|_2^2$$

And add an Autoencoder to ensure that the new observation $x+\delta$ is "realistic" enough. Idea is if it is "realistic" autoencoder will have no problem decoding it

These two optimizations are typically solved via projected gradient descent with additional constraints (e.g., feature ranges, categorical masks)

Finally the problem takes the form of:
$$
\min_{\boldsymbol{\delta}} \ c \cdot \max \left\{ 0, f(\mathbf{x} \odot \boldsymbol{\delta})_t - \max_{i \neq t} f(\mathbf{x} \odot \boldsymbol{\delta})_i + \kappa \right\} + \lambda_1 \cdot \|\boldsymbol{\delta}\|_1 + \lambda_2 \cdot \|\boldsymbol{\delta}\|_2^2 + \lambda_3 \cdot \|\mathbf{x} \odot \boldsymbol{\delta} - AE(\mathbf{x} \odot \boldsymbol{\delta})\|_2^2
$$


# Individual Conditional Expectation (2014)
---
[[paper]](https://arxiv.org/abs/1309.6392)<br>
ICE = Individual Conditional Expectation. For a selected observation $x$ and feature $f_i$ this plot depicts how target depends on possbile values of $f_i$ given that all other features are freezed (ceteri paribas). In other words we slice prediction function at some selected feature

ICE is model agnostic, it works with Black-box models

Name explanation:
- "Individual" because we deal with a single feature at a time
- "Expectation" because we investigate model predictions
- "Conditional" because we fix all other features and possible feature value

Here is an illustration for $(f_1, f_2)$ feature set

<img src="img/ice2.png" width=1000>

ICE shows how a single isolated feature affects the target. It helps to detect:  
- effect magnitude<br>high variablity means high feature importance for a chosen target<br><br>
- shape of the effect<br>linear / non-linear<br><br>
- heterogeneity<br>feature affects target differently along other feature => there is some interaction between them<br><br>

On most production plots many lines are plotted - each corresponds to a single observation in a dataset.
The expectation of these lines (a bold line) gives a Partial Dependency plot - it represents the "averaged" (isolated) effect of the feature

<img src="img/ice_vs_pdp.png" width=500>

__Variants of the plot__
- c-ICE = centered ICE<br>instead of ploting $f(x)$ plot a difference with original observartion<br><br>
- d-ICE = derivative ICE<br>Instead of plotting $f$ plot its derivative $\frac{\partial{f}}{\partial{x}}$ over the feature. It helps highlight areas of rapid prediction change. If it's not constant => there is some interaction with other features

<img src="img/d-ice.png" width=500>

# Partial Dependency Plot (2005)
---
[[paper]]()<br>
PDP = Partial Dependency Plot - a plot that shows how <u>an average</u> prediction depends on possible feature values $x_i$ given that other features are fixed

PDP is just the ICE lines averaged along the analyzed feature

<img src="img/pdp.png" width=500>

# ALE
---
[[paper]](https://arxiv.org/abs/1309.6392)<br>
ALE = Accumulated Local Effect. It's a plot that shows prediction's local change: how it increases or decreases - given that all other features remain fixed, averaged over all available observations and depending on feature value $x$

Averaged and Discretized derivative-ICE gives ALE

Algorithm:
- select a feature to analize
- split feature values into $k$ discrete bins
- calculate prediction change inside each bin $\Delta = f(x_{left}) - f(x_{right})$
- get the averaged difference over across all observations in the dataset
- accumulate those deltas into a continuous plot (integrate from left to right)

<img src="img/ale.png" width=500>



Следующая группа методов работает с понятием концепт. Концепт = вручную введенная фича

# Concept Whitening (2020)

[[paper]](https://arxiv.org/abs/2002.01650)

__Идея:__ давайте вместо батч-нормализации последнего слоя сети делать трансформацию, называемую Concept Whitening. Это добавит интерпретируемости изначально неинтрепретируемым эмбедингам

Трансофрм состоит из двух действий:
- ортогонализация (whitening) вектора активаций<br>запрещаем размазывать фичи по измерениям, хотим так: одна фича = одно измерение
- вращение (alignment)<br>выстраиваем вектор описаний так, чтобы он встал на нашу размеченную шкалу концептов

Алгоритм
1. Учим обычную сеть (pretraining) на разметке классов
2. Строим матрицу ортогонализации $W = {\Lambda}^{1/2} U^T \text{,\quad где \,} z = U \Lambda U^T$ разложение по собственным векторам
3. Строим матрицу поворота Q<br>вектор концепта - это разница между описаниями "концепт есть" / "концепта нету" $q_1 = \frac{\mu_1^+ - \mu_1^-}{\|\mu_1^+ - \mu_1^-\|}$<br>остальные вектора достариваются с учетом требования ортогональности<br>
5. Заменяем BN на CW и дообучаем новую сеть на разметке концептов $Q W (x - \mu)$

Если измерений больше, чем разметили концептов, оставшиеся измерения дозаполняются просто технически ортогональными векторами

Ставится CW-трансформация в один из слоев сети, куда именно - зависит от детализированности концептов (низкоуровневые паттерны можно поближе к началу). Теоретически можно варьировать детализацию и ставить в разные, но это сложно реализовать

# TCAV (2018)
[[paper]](https://arxiv.org/abs/1711.11279)<br>
Testing Concept Activation Vectors - пример post-hoc аналитики, сама модель в рамках никак не модифицируется

__Идея:__ в пространстве эмбедингов давайте построим классификатор, отделяющий примеры обладающие свойством X от примеров без этого свойства. Если при этом какой-то класс Y хорошо отделяется этим классификатором, то считаем, что класс Y хорошо ассоциирован с концептом X

<img src="img/tcav2.png" width=350>

Нужна во-первых разметка на классы (Зебра, Гепард, Акула ...), во-вторых разметка концептов (полостатость, четыре ноги, плавники)


Нормаль к получившейся разделяющей поверхности - это направление концепта (concept vector). Вдоль нее вероятность наличия концепта растет. Величина S - насколько сильно верятность класса растет в анализируемой точке x, если мы движемся вдоль вектора концепта (по сути производная по направлению). 

Агрегированный до класса TCAV Score - это усреднение для всех примеров, доля примеров класса с концептом, которые попадают в правильную полуплоскость

<img src="img/tcav.png" width=750>

a) дан пул собранных экспертами примеров какого-то концепта (например, полосатость)<br>
b) дана обучающая выборка примеров для класса "зебра"<br>
c) обученная сеть<br>
d) TCAV измеряет чувствительность модели к концепту относительно класса
e) CAV вектор - нормаль к линии разделения, вдоль нее увеличивается вер-ть нашего класса. TCAV считает производную по направлению и так оценивает магнитуду увеличения

Результат можно условно выразить в создании коцепт-класс матрицы:
$$\begin{array}{c|ccccc}
 & \text{полосатость} & \text{4 ноги} & \text{снег} & \text{крылья} & \text{клюв} \\
\hline
\text{зебра} & 0.94 & 0.81 & 0.43 & 0.11 & 0.09 \\
\text{волк} & 0.38 & 0.77 & 0.89 & 0.12 & 0.08 \\
\text{орёл} & 0.21 & 0.18 & 0.31 & 0.96 & 0.91 \\
\text{рыба} & 0.44 & 0.11 & 0.22 & 0.19 & 0.14 \\
\end{array}$$

# Concept Bottleneck (2020)
[[paper]](https://arxiv.org/abs/2212.07430)

__Идея:__ давайте обучать модель не только давать правильный ответ, но и генерировать правильные промежуточные эмбединги